In [5]:
# 01_preprocess_arcos.ipynb

import os
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "00_data" / "clean"
PROCESSED_DIR = PROJECT_ROOT / "00_data" / "processed"
DOCS_DIR = PROJECT_ROOT / "00_data" / "docs"


RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)


# Path to the huge ARCOS shipment file you download (see instructions doc)
ARCOS_FILE = RAW_DIR / "arcos_all.tsv"


# -------------------------------------------------------------------
# COLUMN MAP – FILL THESE IN USING THE DATA DICTIONARY
# -------------------------------------------------------------------
COLS = {
    # geography
    "state": "BUYER_STATE",
    "county": "BUYER_COUNTY",  # or county FIPS if that exists
    "zip": "BUYER_ZIP",  # optional
    # timing
    "date": "TRANSACTION_DATE",  # some versions use YYYYMMDD integer
    # drug info
    "drug_name": "DRUG_NAME",
    # or use drug code and filter by codes for oxycodone/hydrocodone
    "drug_code": "DRUG_CODE",
    # shipment volume
    "quantity": "DOSAGE_UNIT",  # or "QUANTITY"
    "mme": "MME",  # if precomputed; else see below
    "strength_mg": "STRENGTH",  # if present
    "mme_factor": "MME_CONVERSION_FACTOR",  # if present
}

In [ ]:
def load_data_dictionary(path: Path | str) -> pd.DataFrame:
    """Load the ARCOS data dictionary for reference."""
    path = Path(path)
    df = pd.read_csv(path)
    return df


def compute_mme(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure we have an MME column.
    If the raw file *already* has MME, this just renames it.
    Otherwise, it tries to compute it using quantity * strength * mme_factor.
    """
    if COLS["mme"] in df.columns:
        df = df.rename(columns={COLS["mme"]: "mme"})
    else:
        qty = df[COLS["quantity"]]
        strength = df[COLS["strength_mg"]]
        factor = df[COLS["mme_factor"]]
        df["mme"] = qty * strength * factor

    return df


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename key columns to simple names used downstream."""
    rename_map = {
        COLS["state"]: "state",
        COLS["county"]: "county",
        COLS["date"]: "date",
        COLS["drug_name"]: "drug_name",
        COLS["drug_code"]: "drug_code",
        COLS["quantity"]: "qty_pills",
    }

    df = df.rename(columns=rename_map)
    return df

In [ ]:
from typing import Iterable


def arcos_chunks(path, chunksize=1_000_000):
    return pd.read_csv(path, sep="\t", chunksize=chunksize, low_memory=False)

In [16]:
from tqdm.auto import tqdm


def process_to_county_year(path: Path) -> pd.DataFrame:
    """
    Reads the ARCOS file in chunks and returns a county-year dataset with:
      - total pills
      - total MME
      - transaction count
    for oxycodone + hydrocodone shipments.
    """
    aggregates = []

    for chunk in tqdm(arcos_chunks(path), desc="Processing ARCOS"):
        # Standardize cols
        chunk = standardize_columns(chunk)

        # Filter to relevant drugs – depending on your file,
        # you may prefer drug_code-based filtering instead of names.
        chunk["drug_name"] = chunk["drug_name"].str.upper().str.strip()
        keep_mask = chunk["drug_name"].isin(["OXYCODONE", "HYDROCODONE"])
        chunk = chunk[keep_mask]

        if chunk.empty:
            continue

        # Parse date → year
        # Some versions store TRANSACTION_DATE as YYYYMMDD integer; adjust as needed.
        chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
        chunk["year"] = chunk["date"].dt.year

        # Drop rows with missing key fields
        chunk = chunk.dropna(subset=["state", "county", "year"])

        # Ensure numeric types
        chunk["qty_pills"] = pd.to_numeric(chunk["qty_pills"], errors="coerce")
        chunk = compute_mme(chunk)
        chunk["mme"] = pd.to_numeric(chunk["mme"], errors="coerce")

        # Group to county-year for this chunk
        grouped = chunk.groupby(["state", "county", "year"], as_index=False).agg(
            total_pills=("qty_pills", "sum"),
            total_mme=("mme", "sum"),
            n_transactions=("drug_name", "size"),
        )
        aggregates.append(grouped)

    if not aggregates:
        raise ValueError("No data processed – check filters / column names.")

    county_year = pd.concat(aggregates, ignore_index=True)

    # Combine across chunks
    county_year = county_year.groupby(["state", "county", "year"], as_index=False).agg(
        total_pills=("total_pills", "sum"),
        total_mme=("total_mme", "sum"),
        n_transactions=("n_transactions", "sum"),
    )

    return county_year

In [ ]:
county_year = process_to_county_year(ARCOS_FILE)

# Save in both CSV and Parquet
out_csv = PROCESSED_DIR / "arcos_county_year.csv"
out_parquet = PROCESSED_DIR / "arcos_county_year.parquet"

county_year.to_csv(out_csv, index=False)
county_year.to_parquet(out_parquet, index=False)

county_year.head()

Processing ARCOS: 0it [00:00, ?it/s]

,state,county,year,total_pills,total_mme,n_transactions
0,AK,ALEUTIANS EAST,2006,300.0,908.10000,1
1,AK,ALEUTIANS EAST,2007,100.0,302.70000,1
2,AK,ALEUTIANS EAST,2019,200.0,605.40000,1
3,AK,ALEUTIANS WEST,2006,26700.0,87308.92500,34
4,AK,ALEUTIANS WEST,2007,33633.0,105802.54725,40
